# AlignLab · Experiment 03 — DPO post-training

Train a **new** preference-optimized LoRA adapter starting from the existing SFT adapter. Designed for Kaggle T4 and Colab. The Kaggle GitHub workflow stages the private SFT adapter as an input dataset. This is a small smoke-test, not a validated improvement claim.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Visible CUDA devices:", os.environ["CUDA_VISIBLE_DEVICES"])

In [ ]:
%pip install -q "transformers>=4.51,<5" "trl>=0.19,<0.27" "peft>=0.15,<0.19" "accelerate>=1.4,<2" "datasets>=3.5,<5" "bitsandbytes>=0.45" "huggingface-hub>=0.30,<1" sentencepiece

In [ ]:
import json
import random
import sys
import time
from datetime import UTC, datetime
from pathlib import Path

import torch
from datasets import load_dataset
from peft import PeftModel, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import DPOConfig, DPOTrainer

assert torch.cuda.is_available(), "Enable a GPU runtime."
assert torch.cuda.device_count() == 1, "Restart runtime; only one GPU should be visible."
print("GPU:", torch.cuda.get_device_name(0))
print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
MODEL_ID = "Qwen/Qwen3-1.7B"
SFT_REPO = "hamedready/alignlab-qwen3-1.7b-qlora"
MAX_TRAIN = 80
MAX_EVAL = 16
MAX_STEPS = 20
MAX_LENGTH = 512
MAX_PROMPT_LENGTH = 256
RUN_ID = datetime.now(UTC).strftime("dpo-%Y%m%dT%H%M%SZ")
OUTPUT_DIR = (
    Path("/kaggle/working/alignlab-dpo")
    if Path("/kaggle/working").exists()
    else Path("/content/alignlab-dpo")
    if Path("/content").exists()
    else Path("./alignlab-dpo")
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output:", OUTPUT_DIR)

## Resolve SFT adapter

For Kaggle, attach the private dataset `hamedhani/alignlab-qlora-adapter`. For Colab, add a read-capable `HF_TOKEN` in Colab Secrets. No secret is required in Kaggle.

In [ ]:
def resolve_sft_adapter():
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        matches = sorted(kaggle_input.rglob("adapter_config.json"))
        matches = [x for x in matches if "alignlab-qlora-adapter" in str(x)]
        if not matches:
            raise FileNotFoundError("SFT adapter dataset is not attached to Kaggle.")
        folder = matches[0].parent
        if not any(
            (folder / name).is_file() for name in ("adapter_model.safetensors", "adapter_model.bin")
        ):
            raise FileNotFoundError("SFT adapter weights missing")
        return str(folder), None
    token = os.getenv("HF_TOKEN")
    if Path("/content").exists() and not token:
        from google.colab import userdata

        token = userdata.get("HF_TOKEN")
    if not token:
        raise RuntimeError("Set HF_TOKEN for private SFT adapter outside Kaggle")
    return SFT_REPO, token.strip()


SFT_SOURCE, HF_READ_TOKEN = resolve_sft_adapter()
print("SFT adapter source:", SFT_SOURCE)

## Preference dataset

UltraFeedback binarized preference examples are independent from the Alpaca training set, but check overlap and dataset licenses before any public benchmark claims. Hold out evaluation examples and never train on them.

In [ ]:
raw = load_dataset("trl-lib/ultrafeedback_binarized", split="train_prefs")
raw = raw.shuffle(seed=SEED).select(range(min(len(raw), MAX_TRAIN + MAX_EVAL)))


def to_text(messages):
    if isinstance(messages, str):
        return messages
    return "\n".join(str(m.get("content", "")) for m in messages)


def convert(row):
    chosen = row["chosen"]
    rejected = row["rejected"]
    if not isinstance(chosen, list) or not isinstance(rejected, list):
        raise ValueError("Expected conversational preference records")
    prompt_messages = [m for m in chosen if m.get("role") != "assistant"]
    chosen_answer = next(
        (m.get("content", "") for m in reversed(chosen) if m.get("role") == "assistant"), ""
    )
    rejected_answer = next(
        (m.get("content", "") for m in reversed(rejected) if m.get("role") == "assistant"), ""
    )
    return {
        "prompt": prompt_messages,
        "chosen": [{"role": "assistant", "content": chosen_answer}],
        "rejected": [{"role": "assistant", "content": rejected_answer}],
    }


data = raw.map(convert, remove_columns=raw.column_names)
data = data.filter(
    lambda x: (
        bool(x["prompt"]) and bool(x["chosen"][0]["content"]) and bool(x["rejected"][0]["content"])
    )
)
assert len(data) > MAX_EVAL, "Not enough valid preference examples"
eval_dataset = data.select(range(MAX_EVAL))
train_dataset = data.select(range(MAX_EVAL, len(data)))
print("Training:", len(train_dataset), "Evaluation:", len(eval_dataset))

## Load quantized SFT model

DPO uses the original SFT adapter as a frozen reference through TRL adapter switching. The trainable adapter is a separate copy named `dpo`.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=quant, device_map={"": 0}, dtype=torch.float16
)
base.config.use_cache = False
base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
model = PeftModel.from_pretrained(
    base, SFT_SOURCE, adapter_name="reference", is_trainable=False, token=HF_READ_TOKEN
)
model.load_adapter(SFT_SOURCE, adapter_name="dpo", is_trainable=True, token=HF_READ_TOKEN)
model.set_adapter("dpo")
for name, param in model.named_parameters():
    if param.requires_grad and param.dtype != torch.float32:
        param.data = param.data.float()
assert any(p.requires_grad for p in model.parameters())
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## Train DPO

TRL versions in the supported range expose `model_adapter_name` and `ref_adapter_name`. We use the SFT reference adapter instead of training against an unrelated base reference.

In [ ]:
args = DPOConfig(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    max_steps=MAX_STEPS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=5e-6,
    beta=0.1,
    max_length=MAX_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="steps",
    save_steps=10,
    save_total_limit=1,
    report_to="none",
    seed=SEED,
    model_adapter_name="dpo",
    ref_adapter_name="reference",
)
assert args.n_gpu == 1
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)
start = time.perf_counter()
result = trainer.train()
train_seconds = time.perf_counter() - start
print("Training completed in seconds:", round(train_seconds, 1))
print(result.metrics)

## Save adapter and report

The output is a PEFT adapter, not a merged base model. GitHub Actions can upload it to a **separate** Hugging Face repository after Kaggle finishes.

In [ ]:
adapter_dir = OUTPUT_DIR / "adapter"
adapter_dir.mkdir(parents=True, exist_ok=True)
model.set_adapter("dpo")
model.save_pretrained(adapter_dir, selected_adapters=["dpo"])
tokenizer.save_pretrained(adapter_dir)
# PEFT may save selected adapters inside a named subdirectory. Normalize it for downstream consumers.
inner = adapter_dir / "dpo"
if (inner / "adapter_config.json").exists():
    import shutil

    for item in inner.iterdir():
        shutil.move(str(item), str(adapter_dir / item.name))
    inner.rmdir()
assert (adapter_dir / "adapter_config.json").is_file()
assert any((adapter_dir / x).is_file() for x in ("adapter_model.safetensors", "adapter_model.bin"))
eval_metrics = trainer.evaluate()
report = {
    "experiment": "03_dpo_training",
    "run_id": RUN_ID,
    "base_model": MODEL_ID,
    "sft_adapter": SFT_REPO,
    "preference_dataset": "trl-lib/ultrafeedback_binarized",
    "training_examples": len(train_dataset),
    "evaluation_examples": len(eval_dataset),
    "max_steps": MAX_STEPS,
    "train_seconds": train_seconds,
    "train_metrics": result.metrics,
    "eval_metrics": eval_metrics,
    "note": "Smoke-test run; no response-quality improvement claim without held-out evaluation.",
}
report_path = OUTPUT_DIR / "dpo_report.json"
report_path.write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")
print("Adapter:", adapter_dir, "Report:", report_path)

In [ ]:
import shutil

archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Downloadable archive:", archive)